# **Laboratorio Groq**

Este laboratorio se enfoca en la implementación de agentes utilizando los modelos de alto rendimiento de **Groq**.

### **Instrucciones para configurar Groq:**

1. **Obtener la API Key:**
   - Dirígete a la [Groq Console](https://console.groq.com/).
   - Crea una cuenta si aún no la tienes.
   - En el menú lateral, selecciona **API Keys** y genera una nueva llave.

2. **Configuración del Modelo:**
   - En este laboratorio utilizaremos el modelo llama-3.3-70b-versatile por su equilibrio entre velocidad y capacidad de razonamiento.
   - Asegúrate de tener la llave configurada en tus secretos de Colab o en un archivo .env como GROQ_API_KEY.


## Introducción a los Agentes de IA

Un **Agente de Inteligencia Artificial** es un sistema capaz de **percibir su entorno, tomar decisiones y ejecutar acciones** para cumplir un objetivo. A diferencia de un LLM tradicional (como ChatGPT, que solo genera texto en base a un prompt), un agente utiliza el modelo de lenguaje como su **cerebro o motor de razonamiento**, permitiéndole planificar pasos y usar **Herramientas (Tools)** externas (bases de datos, APIs, calculadoras) para obtener información actualizada y resolver problemas complejos.

### Componentes Principales de un Agente:
1. **El Cerebro (LLM)**: Procesa la solicitud, razona qué debe hacer y decide qué herramientas usar.
2. **Herramientas (Tools)**: Funciones que el agente puede invocar. Actúan como los brazos y ojos del agente.
3. **Memoria**: Capacidad de recordar interacciones pasadas.
4. **Marco de Razonamiento (Ej. ReAct)**: Un patrón donde el agente alterna entre **Pensar** (Thought) y **Actuar** (Action).

In [ ]:
%reset -f


### 1. Preparación del Entorno
En esta sección instalaremos y configuraremos las librerías necesarias. Utilizaremos **LangChain**, un framework diseñado específicamente para construir aplicaciones y agentes basados en modelos de lenguaje.

In [ ]:
# !kill -9 -1  # Descomentar solo si se ejecuta en Google Colab para reiniciar el entorno

In [ ]:
!pip install -U -q langchain==0.3.7 langchainhub==0.1.15 langchain_groq langchain_community python-dotenv unidecode duckduckgo-search


In [ ]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# Cargar variables de entorno desde un archivo .env si existe
load_dotenv()

# Cargar la API Key de Groq
try:
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
except ImportError:
    api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    raise ValueError("La variable GROQ_API_KEY no está definida. Añádela en los secretos de Colab o en un archivo .env local.")

os.environ["GROQ_API_KEY"] = api_key

# Inicializamos el modelo (cerebro del agente)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [ ]:
from langchain.tools import tool
from unidecode import unidecode

@tool
def get_weather_forecast(city: str) -> str:
    """Devuelve el pronóstico del tiempo para una ciudad específica."""
    c = unidecode(city.lower())
    if "tokyo" in c:
        return "El pronóstico para Tokyo es de 25°C con sol y nubes."
    elif "paris" in c:
        return "El pronóstico para París es de 18°C y está lloviendo."
    else:
        return f"No tengo información del tiempo para {city}."


@tool
def get_flight_details(query: str) -> str:
    """Obtiene detalles de vuelos dados en una frase completa (por ejemplo 'vuelo de Tokyo a Sydney')."""
    q = unidecode(query.lower())
    if "new york" in q and "london" in q:
        return "Hay un vuelo de New York a Londres por $600 USD que dura 7 horas."
    elif "tokyo" in q and "sydney" in q:
        return "Hay un vuelo de Tokyo a Sydney por $800 USD que dura 9 horas."
    else:
        return "Lo siento, no encontré información para ese vuelo."

tools = [get_weather_forecast, get_flight_details]


In [ ]:
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor
# Cargamos el prompt ReAct desde LangChain Hub
prompt = hub.pull("hwchase17/react")
# Creamos el agente con las herramientas y el LLM
agent = create_react_agent(llm, tools, prompt)
# Orquestador del ciclo ReAct
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# Prueba 1: Una sola herramienta
print("--- Ejecutando Prueba 1---")
agent_executor.invoke({"input": "Quiero saber cómo estará el tiempo en Paris."})
# Prueba 2: Múltiples herramientas
print("\n--- Ejecutando Prueba 2---")
agent_executor.invoke({"input": "Necesito planificar un viaje de Tokyo a Sydney. ¿Cuánto cuesta y dura el vuelo, y cómo estará el tiempo en Tokyo?"})

### 6. Ampliando las Capacidades del Agente
Para que un agente aprenda a hacer algo nuevo, simplemente debemos programar una nueva herramienta y dársela. Aquí agregaremos una herramienta para consultar la **moneda local** de una ciudad.

In [ ]:
from langchain.tools import tool
from unidecode import unidecode

@tool
def get_weather_forecast(city: str) -> str:
    """Devuelve el pronóstico del tiempo para una ciudad específica."""
    c = unidecode(city.lower())
    if "tokyo" in c:
        return "El pronóstico para Tokyo es de 25°C con sol y nubes."
    elif "paris" in c:
        return "El pronóstico para París es de 18°C y está lloviendo."
    else:
        return f"No tengo información del tiempo para {city}."


@tool
def get_flight_details(query: str) -> str:
    """Obtiene detalles de vuelos dados en una frase completa (por ejemplo 'vuelo de Tokyo a Sydney')."""
    q = unidecode(query.lower())
    if "new york" in q and "london" in q:
        return "Hay un vuelo de New York a Londres por $600 USD que dura 7 horas."
    elif "tokyo" in q and "sydney" in q:
        return "Hay un vuelo de Tokyo a Sydney por $800 USD que dura 9 horas."
    else:
        return "Lo siento, no encontré información para ese vuelo."

@tool
def get_local_currency(city: str)-> str:
  "Devuelve la moneda local."
  c = unidecode(city.lower())
  if "tokyo" in c:
    return "La moneda local en Tokyo es el Yen Japonés (JPY)."
  elif "paris" in c:
    return "La moneda local en París es el Euro (EUR)."
  else:
    return f"No tengo información de la moneda local para {city}."

tools = [get_weather_forecast, get_flight_details, get_local_currency]

In [ ]:
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor
# Cargamos el prompt ReAct desde LangChain Hub
prompt = hub.pull("hwchase17/react")
# Creamos el agente con las herramientas y el LLM
agent = create_react_agent(llm, tools, prompt)
# Orquestador del ciclo ReAct
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# Prueba 1: Una sola herramienta (clima)
print("\n--- Ejecutando Prueba 1 ---")
agent_executor.invoke({"input": "Quiero saber cómo estará el tiempo en Paris."})

# Prueba 2: Múltiples herramientas (vuelos + clima)
print("\n--- Ejecutando Prueba 2 ---")
agent_executor.invoke({"input": "Necesito planificar un viaje de Tokyo a Sydney. ¿Cuánto cuesta y dura el vuelo, y cómo estará el tiempo en Tokyo?"})

# Prueba 3: Nueva herramienta (moneda local)
print("\n--- Ejecutando Prueba 3 ---")
agent_executor.invoke({"input": "¿Cuál es la moneda local en Sydney?"})


### 7. Proyecto Final: Agente de Soporte Técnico y Presupuesto Dinámico

Este ejercicio integra búsqueda real en internet, lógica de inventario y cálculos matemáticos para demostrar la utilidad de un agente en un flujo de trabajo real.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

# 1. Herramienta de búsqueda real
search = DuckDuckGoSearchRun()

@tool
def calculate_pc_budget(components_list: str) -> str:
    """Calcula el presupuesto total de una lista de componentes (ej: 'SSD:100, RAM:50')."""
    try:
        items = components_list.split(",")
        total = sum(float(item.split(":")[1]) for item in items)
        return f"El total de los componentes es ${total:.2f} USD. Con un impuesto del 19% (IVA), el total es ${total * 1.19:.2f} USD."
    except Exception as e:
        return f"Error al calcular: {str(e)}. Usa el formato 'Componente:Precio'."

@tool
def get_current_stock_status(component_name: str) -> str:
    """Verifica si un componente específico está disponible en bodega."""
    inventory = {"rtx 4080": "Agotado", "rtx 4070": "Disponible", "ssd 1tb": "Disponible", "i9 14900k": "Disponible"}
    status = inventory.get(component_name.lower(), "No encontrado en inventario local")
    return f"Estado de {component_name}: {status}"

# Actualizamos las herramientas para el nuevo agente
tools = [search, calculate_pc_budget, get_current_stock_status]

# Re-configuramos el agente
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# PRUEBA DEL PROYECTO:
print("\n--- Ejecutando Proyecto Final ---")
query = "Busca el precio promedio de una RTX 4070, verifica si hay stock y calcula el total sumando un SSD 1TB de 80 USD incluyendo el IVA del 19%."
agent_executor.invoke({"input": query})

### 🏆 Desafío Autónomo: Agente Ingeniero e Importador de AliExpress

Para poner a prueba el verdadero poder de razonamiento de un agente, tu tarea es crear un asistente que **diseñe una solución a un problema e interactúe con APIs verdaderas** para calcular un presupuesto internacional.

**Herramientas y Tecnologías Reales Sugeridas:**
En lugar de simular datos, ¡te retamos a construir herramientas que consulten el mundo real!

1. **Consultor de Ingeniería (Investigación Web):** Integra `DuckDuckGoSearchRun` de `langchain_community.tools` para que el agente investigue qué componentes electrónicos específicos (ej. tipos de sensores, placas) resuelven el problema planteado.
2. **Scraper de Precios (AliExpress / E-commerce):** Crea una herramienta `@tool` que reciba el nombre de un componente y extraiga su precio real en USD. Puedes intentar usar `requests` y `BeautifulSoup` para hacer web scraping básico, o registrarte para obtener una API gratuita (ej. en RapidAPI) de e-commerce.
3. **Conversor de Divisas en Tiempo Real:** Implementa una herramienta que haga una petición HTTP (`requests.get`) a una API pública y gratuita (como `https://api.exchangerate-api.com/v4/latest/USD`) para obtener la tasa de cambio actual al día de hoy y realizar la conversión.
4. **Calculadora de Costos (Opcional):** Para sumar varios componentes e impuestos, puedes darle al agente una calculadora precisa usando integraciones como `LLMMathChain` o construir una propia.

---
### 🛠️ Metodología Recomendada de Desarrollo
Construir un agente real puede ser complejo si intentas hacer todo a la vez. Sigue este enfoque iterativo:

**Fase 1: Prueba Unitaria de Herramientas (Aislamiento)**
* No pienses en el agente todavía. Escribe tus funciones de Python normales (ej. `def obtener_divisa():`) y ejecútalas solas con un `print()` para asegurarte de que la conexión a la API o el scraping funciona y extrae la información correcta.
* Una vez que la función clásica sirva, agrégale el decorador `@tool` y **escribe un Docstring (comentario de triple comilla) extremadamente detallado**. El agente leerá este Docstring para decidir cuándo usar la herramienta y qué parámetros pasarle.

**Fase 2: Ensamblaje del Agente**
* Crea la lista `tools = [...]` con tus herramientas comprobadas.
* Inicializa tu `AgentExecutor` asegurándote de colocar `verbose=True`. Esto es vital: te permitirá leer la cadena de "Pensamiento" (Thought) del agente. Si falla, el Thought te dirá si se confundió con el formato de entrada o si no entendió el propósito de una herramienta.

**Fase 3: Pruebas Incrementales del Prompt**
* **Nivel 1:** Pruébalo con algo simple: *"¿A cómo está el dólar hoy?"*. Verifica que invoque correctamente el conversor.
* **Nivel 2:** Incrementa la dificultad: *"Busca un ESP32 en AliExpress y dime su precio"*. Verifica tu scraper.
* **Nivel 3:** Lanza el desafío completo.
---

**El objetivo final del agente debe ser resolver un prompt complejo e integrado como:**
> *"Tengo un problema: necesito construir un sistema económico para regar mis plantas automáticamente cuando la tierra esté seca, y que me avise por WiFi. Investiga qué componentes electrónicos básicos necesito para armarlo, busca sus precios reales actuales en internet (USD), conviértelos a mi moneda local usando la tasa de cambio de hoy, y entrégame un presupuesto total detallado."*

¡Diseña las herramientas, sigue la metodología y deja que tu agente se enfrente a internet en la siguiente celda!

In [ ]:
# Tu código para el desafío de AliExpress aquí
# 1. Define las nuevas herramientas


# 2. Crea la lista de tools


# 3. Configura el agente y ejecútalo